# Data Preprocessing

In [4]:
import os
import json
import glob
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_TRAIN_DIR = os.path.join(DATA_DIR, "raw_json", "train", "extracted")
RAW_TEST_DIR = os.path.join(DATA_DIR, "raw_json", "val", "extracted")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

MAX_WORKERS = 8
print(f"Using {MAX_WORKERS} worker threads for parallel preprocessing.")

Using 8 worker threads for parallel preprocessing.


## 1. Feature Extraction Definitions

In [5]:
UPPER_BODY_INDICES = [0, 1, 2, 3, 4, 5, 6, 7]

H_PARENTS = [0, 1, 2, 3, 0, 5, 6, 7, 0, 9, 10, 11, 0, 13, 14, 15, 0, 17, 18, 19]
H_CHILDREN = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
UB_PARENTS = [1, 1, 2, 3, 1, 5, 6]
UB_CHILDREN = [0, 2, 3, 4, 5, 6, 7]

def extract_openpose_joint(pose3d, joint_idx):
    base = joint_idx * 4
    if len(pose3d) >= base + 3:
        return np.array([pose3d[base], pose3d[base+1], pose3d[base+2]], dtype=np.float32)
    return np.zeros(3, dtype=np.float32)

def extract_hand_coords(hand3d):
    if len(hand3d) == 84 and any(hand3d):
        return np.array([[hand3d[i], hand3d[i+1], hand3d[i+2]] for i in range(0, 84, 4)], dtype=np.float32)
    return np.zeros((21, 3), dtype=np.float32)

def compute_25d_features(coords, parents, children, angle_v1, angle_v2):
    coords = np.array(coords, dtype=np.float32)
    if coords.sum() == 0:
        return np.zeros((len(parents)*3 + len(angle_v1)), dtype=np.float32)
    
    p_coords = coords[parents]
    c_coords = coords[children]
    bones = c_coords - p_coords
    
    dx, dy, dz = bones[:, 0], bones[:, 1], bones[:, 2]
    norm2d = np.sqrt(dx**2 + dy**2) + 1e-6
    u_x, u_y = dx / norm2d, dy / norm2d
    
    v1, v2 = bones[angle_v1, :2], bones[angle_v2, :2]
    dot = np.sum(v1 * v2, axis=1)
    n1, n2 = np.linalg.norm(v1, axis=1), np.linalg.norm(v2, axis=1)
    angles = np.arccos(np.clip(dot / (n1 * n2 + 1e-6), -1.0, 1.0))
    
    return np.concatenate([u_x, u_y, dz, angles])

def parse_word_dir(dir_path, sequence_length=30):
    try:
        json_files = sorted(glob.glob(os.path.join(dir_path, "*.json")))
        if not json_files: return None

        frames_feats = []
        for jf in json_files:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
            people = data.get("people", {})
            if isinstance(people, list) and len(people) > 0: people = people[0]

            pose3d = people.get("pose_keypoints_3d", [])
            r3d, l3d = people.get("hand_right_keypoints_3d", []), people.get("hand_left_keypoints_3d", [])

            upper_joints = np.array([extract_openpose_joint(pose3d, j) for j in UPPER_BODY_INDICES], dtype=np.float32)
            rh, lh = extract_hand_coords(r3d), extract_hand_coords(l3d)

            neck = upper_joints[1].copy()
            upper_joints -= neck; rh -= neck; lh -= neck

            swidth = np.linalg.norm(upper_joints[5] - upper_joints[2])
            if swidth < 1e-6: swidth = 1.0
            
            upper_joints /= swidth; rh /= swidth; lh /= swidth

            r_feat = compute_25d_features(rh, H_PARENTS, H_CHILDREN, [0, 4, 8], [1, 5, 9])
            l_feat = compute_25d_features(lh, H_PARENTS, H_CHILDREN, [0, 4, 8], [1, 5, 9])
            ub_feat = compute_25d_features(upper_joints, UB_PARENTS, UB_CHILDREN, [2, 5, 1], [3, 6, 4])
            
            frames_feats.append(np.concatenate([r_feat, l_feat, ub_feat]))

        frames_arr = np.array(frames_feats, dtype=np.float32)
        total_f = len(frames_arr)

        if total_f < sequence_length:
            pad = np.tile(frames_arr[-1:], (sequence_length - total_f, 1))
            frames_arr = np.vstack([frames_arr, pad])
        elif total_f > sequence_length:
            frames_arr = frames_arr[np.linspace(0, total_f - 1, sequence_length, dtype=int)]

        return frames_arr
    except Exception:
        return None

## 2. Process Data Using core_label_map.json

In [ ]:
core_label_map_path = os.path.join(PROCESSED_DIR, "core_label_map.json")
with open(core_label_map_path, "r", encoding="utf-8") as f:
    core_label_map = json.load(f)

def process_dir(w_dir):
    try:
        folder_name = os.path.basename(w_dir)
        parts = folder_name.split("_")
        word_id = parts[2] if len(parts) > 2 else folder_name
        if word_id in core_label_map:
            seq = parse_word_dir(w_dir)
            if seq is not None:
                return core_label_map[word_id], seq
    except:
        pass
    return None, None

def build_dataset(raw_dir, prefix):
    word_dirs = glob.glob(os.path.join(raw_dir, "**", "NIA_SL_WORD*"), recursive=True)
    print(f"Found {len(word_dirs)} raw directories in {prefix}. Processing only core classes...")
    
    X, y = [], []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        valid_dirs = [d for d in word_dirs if (os.path.basename(d).split('_')[2] if len(os.path.basename(d).split('_')) > 2 else os.path.basename(d)) in core_label_map]
        print(f'Filtered {len(word_dirs)} down to {len(valid_dirs)} core directories.\n')
        results = list(tqdm(executor.map(process_dir, valid_dirs), total=len(valid_dirs)))

    for res in results:
        if res[0] is not None:
            y.append(res[0])
            X.append(res[1])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    
    np.save(os.path.join(PROCESSED_DIR, f"X_{prefix}.npy"), X)
    np.save(os.path.join(PROCESSED_DIR, f"y_{prefix}.npy"), y)
    print(f"{prefix.capitalize()} Exported -> X: {X.shape}, y: {y.shape}")

build_dataset(RAW_TRAIN_DIR, "train")
build_dataset(RAW_TEST_DIR, "test")
print("Uncomment above lines to run processing if raw JSON directories exist.")
+

Found 18925804 raw directories in train. Processing only core classes...
Filtered 18925804 down to 646054 core directories.



100%|██████████| 646054/646054 [09:23<00:00, 1145.79it/s] 


Train Exported -> X: (5600, 30, 150), y: (5600,)
Found 57519 raw directories in test. Processing only core classes...
Filtered 57519 down to 1120 core directories.



100%|██████████| 1120/1120 [00:00<00:00, 20260.15it/s]

Test Exported -> X: (0,), y: (0,)
Uncomment above lines to run processing if raw JSON directories exist.
